# Datová analytika - vytvoření sázkových kurzů
## Cíl

Úkolem bude získat výsledky NHL z veřejně dostupných zdrojů a následně určit počáteční sázkové kurzy.

Čtyři hlavní částí tohoto notebooku:

- **1. Stažení dat** – stáhnutí dat ze stránky Scrape This Site (část 1),

- **2. Zpracování dat** – analýza HTML a připrava data pro analýzu (část 2),

- **3. Analýza dat** – explorační analýzu dat (část 3),

- **4. Výsledek** – počáteční sázkové kurzy (část 4).

---

## Požadované knihovny

- **requests** – stažení HTML obsahu stránek se zápasy,

- **BeautifulSoup** – zpracování nestrukturovaných dat (HTML kódu) do tabulární podoby (DataSet),

- **Pandas** – provádění transformací dat,

- **Matplotlib** – prezentaci výsledků.

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from glob import glob
import json
from glob import glob
import matplotlib.pyplot as plt

## 1. Stažení dat
- Stažení tabulek do html souborů

In [ ]:
# URL
url = 'https://www.scrapethissite.com/pages/forms/?per_page=100'

try:
    # Vytvor request
    response = requests.get(url)
    response.raise_for_status()

    # BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')
    url_list = [i.text.strip() for i in soup.select(
        '.pagination a:not([aria-label="Next"])')]

    for index in url_list:
        table_url = f"https://www.scrapethissite.com/pages/forms/?page_num={index}&per_page=100"
        table_response = requests.get(table_url)
        table_response.raise_for_status()

        table_soup = BeautifulSoup(table_response.text,'html.parser')

        with open(f"./data/raw/hockey_table{index.zfill(2)}.html", "w", encoding="utf-8") as hpage:
            hpage.write(table_response.text)

        sleep(1)

except requests.exceptions.HTTPError as err:
    print(f"HTTP chyba nastala: {err}")
except Exception as err:
    print(f"Jiná chyba: {err}")
    

### Shrnutí
 
Stažení surových dat ze zdroje snížilo riziko problémů vyplývajících z aktualizací stránek během procesu extrakce. Tato metoda má i další výhodu: umožňuje snadný přístup k datům v jejich původní podobě, což je klíčové v případě nutnosti opětovného zpracování.
 
V dalším kroku se zaměříme na extrakci potřebných informací z `html` stránek, což je nezbytné pro provedení analýzy.

## 2. Zpracování dat
Uložení informací(`/data/raw`) o hokejových týmech do formátu JSON.. Extrahovaná data budou zahrnovat:

- Název týmu (`Team Name`),

- Rok (`Year`),

- Počet výher (`Wins`),

- Počet proher (`Losses`),

- Počet proher v prodloužení (`OT Losses` – Overtime Losses),

- Procento výher (`Win %`),

- Počet vstřelených gólů (`Goals For (GF)`),

- Počet inkasovaných gólů (`Goals Against (GA)`),

- Rozdíl skóre (`+ / -`).

Každý získaný záznam by měl být uspořádán do slovníku ve struktuře uvedené níže a poté přidán do seznamu výsledků:

```python
{
    'Team Name': 'Boston Bruins',
    'Year': '1990',
    'Wins': '44',
    'Losses': '24',
    'OT Losses': '',
    'Win %': '0.55',
    'Goals For (GF)': '299',
    'Goals Against (GA)': '264',
    '+ / -': '35'
}
```


In [ ]:
files = glob("./data/raw/*.html")

column_names = []
final_team_list = []

with open(files[0],"r") as f:
    soup = BeautifulSoup(f.read(),"html.parser")
    column_names = [item.text.strip() for item in soup.select(".table th")]

for file in files:
    team_list = []
    with open(file, "r") as f:
        soup = BeautifulSoup(f.read(),'html.parser')
        team_list = [team for team in soup.select("tr.team")]
        
        for team in team_list:
            team_values = [team_value.text.strip() for team_value in team.select("td")]
            team_dict = {}
            for column_name, value in zip(column_names, team_values):
                team_dict[column_name] = value

            final_team_list.append(team_dict)

with open("./data/raw/hockey_teams.json","w",encoding="utf-8") as f:
    json.dump(final_team_list, f, indent=4)

## 3. Analýza dat

In [10]:
df_raw = pd.read_json('./data/raw/hockey_teams.json')
df = df_raw.copy()
df.head()

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Edmonton Oilers,1994,17,27,,0.354,136,183,-47
1,Florida Panthers,1994,20,22,,0.417,115,127,-12
2,Hartford Whalers,1994,19,24,,0.396,127,141,-14
3,Los Angeles Kings,1994,16,23,,0.333,142,174,-32
4,Montreal Canadiens,1994,18,23,,0.375,125,148,-23


## Předběžná transformace dat

### Standardizace názvů sloupců

V současnosti naše datová sada obsahuje původní názvy sloupců, které by z technického hlediska neměly obsahovat mezery nebo jiné speciální znaky.

V této části standardizujeme a zjednodušíme názvy sloupců, aby bylo následné zpracování dat snazší. Použijeme následující mapování názvů:

- Team Name -> `team`

- Year -> `season`

- Wins -> `victories`

- Losses -> `defeats`

- OT Losses -> `overtime_defeats`

- Win % -> `victory_percentage`

- Goals For (GF) -> `scored_goals`

- Goals Against (GA) -> `received_goals`

- +/- -> `goal_difference`

In [13]:
df.columns = ["team", "season", "victories", "defeats", "overtime_defeats",
              "victory_percentage", "scored_goals", "received_goals", "goal_difference",]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   team                582 non-null    str    
 1   season              582 non-null    int64  
 2   victories           582 non-null    int64  
 3   defeats             582 non-null    int64  
 4   overtime_defeats    582 non-null    str    
 5   victory_percentage  582 non-null    float64
 6   scored_goals        582 non-null    int64  
 7   received_goals      582 non-null    int64  
 8   goal_difference     582 non-null    int64  
dtypes: float64(1), int64(6), str(2)
memory usage: 41.1 KB


Odtranění prázdných hodnot

In [21]:
# Projdi vsechny sloupce a vypis unikatni hodnoty
import pandas as pd

# Projdeme každý sloupec a rovnou vypíšeme výsledek
for col in df.columns:
    print(f"--- Sloupec: {col} ---")
    print(df[col].value_counts())
    print("\n")  # Přidá prázdný řádek pro přehlednost

--- Sloupec: team ---
team
Edmonton Oilers            21
Los Angeles Kings          21
Montreal Canadiens         21
New Jersey Devils          21
New York Islanders         21
New York Rangers           21
Philadelphia Flyers        21
Pittsburgh Penguins        21
St. Louis Blues            21
Toronto Maple Leafs        21
Vancouver Canucks          21
Washington Capitals        21
Boston Bruins              21
Buffalo Sabres             21
Calgary Flames             21
Chicago Blackhawks         21
Detroit Red Wings          21
San Jose Sharks            20
Ottawa Senators            19
Tampa Bay Lightning        19
Florida Panthers           18
Dallas Stars               18
Colorado Avalanche         16
Phoenix Coyotes            15
Carolina Hurricanes        14
Nashville Predators        13
Mighty Ducks of Anaheim    12
Atlanta Thrashers          11
Columbus Blue Jackets      11
Minnesota Wild             11
Hartford Whalers            7
Winnipeg Jets               7
Anaheim Ducks